In [ ]:
# Colab 請選 2026.07；安裝到獨立環境，避免更改核心已載入的 NumPy。
from pathlib import Path
import subprocess
import sys
import tempfile

assert sys.version_info[:3] == (3, 12, 13), "請使用書籍或 Colab 2026.07 的 Python 3.12.13"
python = sys.executable
if "google.colab" in sys.modules:
    %pip -q install virtualenv
    env = Path("/content/ch06-book-env")
    if not (env / "bin/python").is_file():
        subprocess.run([python, "-m", "virtualenv", "-q", str(env)], check=True)
    python = str(env / "bin/python")
    pins = ["deepchem==2.8.1.dev20260821122258", "numpy==1.26.4", "scipy==1.17.1",
            "rdkit==2026.9.1", "scikit-learn==1.9.1", "pandas==3.0.6", "matplotlib==3.11.2"]
    subprocess.run([python, "-m", "pip", "install", "-q", *pins], check=True)
    subprocess.run([python, "-m", "pip", "check"], check=True)
work = tempfile.TemporaryDirectory()
script = Path(work.name) / "06-02-morgan-fingerprints.py"

取 ESOL 的苯甲醚，追蹤 Morgan 環境折疊後的位置。首次執行需連網下載 ESOL。

In [ ]:
%%writefile "{script}"
from pathlib import Path
import os
import tempfile

import deepchem as dc
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from rdkit import Chem
from rdkit.Chem import Draw, rdFingerprintGenerator

# ESOL：1,128 筆，實測 log10 溶解度（mol/L），Delaney DOI 10.1021/ci034243x。
# 這裡只讀結構；保留原始標籤，沒有訓練模型。
np.random.seed(42)
cache = Path(os.environ.get("DEEPCHEM_DATA_DIR", Path(tempfile.gettempdir()) / "cheminfo-datasets"))
cache.mkdir(parents=True, exist_ok=True)
tasks, splits, _ = dc.molnet.load_delaney(
    featurizer=dc.feat.RawFeaturizer(smiles=True), splitter="scaffold",
    transformers=[], data_dir=str(cache), reload=False)
assert sum(map(len, splits)) == 1128
records = {str(s).strip() for split in splits for s in split.ids}
figure_dir = Path("../../figures") if Path("../../outline.yml").is_file() else Path("figures")
figure_dir.mkdir(parents=True, exist_ok=True)

smiles = "COc1ccccc1"
assert smiles in records
mol = Chem.MolFromSmiles(smiles)
lengths = [16, 64, 128, 2048]
rows = {}
for radius in [0, 1, 2, 3]:
    generator = rdFingerprintGenerator.GetMorganGenerator(radius=radius)
    raw = generator.GetSparseCountFingerprint(mol).GetNonzeroElements()
    occupied = []
    for size in lengths:
        folded = rdFingerprintGenerator.GetMorganGenerator(radius=radius, fpSize=size)
        on = set(folded.GetFingerprint(mol).GetOnBits())
        assert on == {key % size for key in raw}
        occupied.append(len(on))
    rows[radius] = (len(raw), occupied)
    print(radius, sum(raw.values()), len(raw), occupied)

info = rdFingerprintGenerator.AdditionalOutput()
info.CollectBitInfoMap()
generator = rdFingerprintGenerator.GetMorganGenerator(radius=2)
generator.GetSparseCountFingerprint(mol, additionalOutput=info)
for key, centers in sorted(info.GetBitInfoMap().items()):
    if key % 16 == 1:
        print(key, centers)

fig, (structure, ax) = plt.subplots(1, 2, figsize=(8.7, 3.9),
    gridspec_kw={"width_ratios": [1, 1.7]}, layout="constrained")
numbered = Chem.Mol(mol)
for atom in numbered.GetAtoms():
    atom.SetProp("atomNote", str(atom.GetIdx()))
structure.imshow(Draw.MolToImage(numbered, size=(390, 350)))
structure.set_title("Anisole | atom index")
structure.axis("off")
for radius, (distinct, values) in rows.items():
    ax.plot(range(len(lengths)), values, marker="o", label=f"r = {radius}  ({distinct} IDs)")
ax.set_xticks(range(len(lengths)), lengths)
ax.set_yticks(range(0, 19, 3))
ax.set(xlabel="Fingerprint size (bits)", ylabel="Occupied bits", ylim=(0, 19))
ax.legend(ncol=2, fontsize=8, loc="lower right")
fig.savefig(figure_dir / "06-02-morgan-fingerprints.png", dpi=150)


In [ ]:
from IPython.display import Image, display

subprocess.run([python, str(script)], check=True)
figure_dir = Path("../../figures") if Path("../../outline.yml").is_file() else Path("figures")
display(Image(filename=str(figure_dir / "06-02-morgan-fingerprints.png")))